# Lesson 17: Routing, Scaling, Custom Transports & Adversarial Debate Agents

In this notebook, you will:
1. Build a shared `FastMCP` tool server for fact-checking.
2. Run a 3-agent **Adversarial Debate** (`ForAgent`, `AgainstAgent`, `JudgeAgent`) powered by `gemini-2.5-flash` (with graceful offline fallback).
3. Inspect how shared MCP evidence shapes the Judge's final verdict.

In [ ]:
!pip install -q -U mcp google-genai

In [ ]:
import os
from google import genai
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("SharedDebateTools")

@mcp.tool()
def lookup_benchmark(topic: str) -> str:
    """Shared factual lookup tool for both debate agents."""
    facts = {
        "stateless_mcp": "Stateless HTTP eliminates sticky sessions, while write tools still require idempotency keys to prevent duplicate side effects on retry."
    }
    return facts.get(topic, "No benchmark found.")

evidence = lookup_benchmark("stateless_mcp")
print("Shared MCP evidence:", evidence)

api_key = os.environ.get("GEMINI_API_KEY")
if api_key:
    client = genai.Client(api_key=api_key)
    for_arg = client.models.generate_content(model="gemini-2.5-flash", contents=f"Argue FOR stateless MCP using: {evidence}").text
    against_arg = client.models.generate_content(model="gemini-2.5-flash", contents=f"Argue AGAINST naive stateless MCP using: {evidence}").text
    judge = client.models.generate_content(model="gemini-2.5-flash", contents=f"Judge these two arguments:\nFOR: {for_arg}\nAGAINST: {against_arg}").text
    print("Judge Verdict:\n", judge)
else:
    print("[Offline Demo] Judge Verdict: Adopt stateless MCP 2026-07-28 paired with idempotency keys for all write tools.")

### Challenge Exercise
Implement a custom in-memory `asyncio.Queue` transport adapter that passes JSON-RPC strings between a producer coroutine and a consumer coroutine.